# A/B Testing: Checkout Conversion Analytics Pipeline
This notebook evaluates a checkout UI experiment using a SQL data pipeline (DuckDB) and Python statistical testing.

In [ ]:
!pip install duckdb pandas statsmodels

import pandas as pd
import duckdb
import scipy.stats as stats
from statsmodels.stats.proportion import proportions_ztest
import matplotlib.pyplot as plt
import seaborn as sns

### 1. Data Pipeline (Extraction & Cleaning)
Using DuckDB to extract the dataset, remove duplicate user entries, handle routing errors, and aggregate the raw conversion numbers.

In [ ]:
query="""
WITH CleanData AS(
    SELECT *,
    ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY timestamp DESC) as user_rank
    FROM 'ab_data.csv'
    WHERE ("group"='control' AND landing_page='old_page')
    OR ("group"='treatment' AND landing_page='new_page')
),
Uniqueusers AS(
    SELECT * FROM CleanData WHERE user_rank=1
)

SELECT
    "group",
    COUNT(user_id) as user_count,
    SUM(converted) as converted_count,
    ROUND((SUM(converted)*100.0)/COUNT(user_id), 2) as conversion_rate
FROM Uniqueusers
GROUP BY "group";
"""

# Execute the query using DuckDB and fetch the results
summary_df = duckdb.query(query).df()
print("--- SQL AGGREGATION RESULTS ---")
print(summary_df)

### 2. Data Validation (Sample Ratio Mismatch)
Running a Chi-Square Goodness of Fit test to mathematically prove the 50/50 traffic split is valid and free of routing bugs.

In [ ]:
# Extract numbers for statistical testing
treatment_data = summary_df[summary_df["group"]=='treatment'].iloc[0]
control_data = summary_df[summary_df["group"]=='control'].iloc[0]

observed = [control_data['user_count'], treatment_data['user_count']]
total_users = sum(observed)
expected = [total_users / 2, total_users / 2]

chi2_stat, p_value_srm = stats.chisquare(f_obs=observed, f_exp=expected)

print("--- Sample Ratio Mismatch (SRM) Check ---")
print(f"SRM P-Value: {p_value_srm:.4f}")
if p_value_srm < 0.05:
    print("Result: SRM DETECTED. Do not trust the A/B test results.\n")
else:
    print("Result: No SRM detected. Data is safe to analyze.\n")

### 3. Statistical Analysis (Z-Test)
Testing if the difference in checkout conversion rates is statistically significant.

In [ ]:
successes = [treatment_data['converted_count'], control_data['converted_count']]
trials = [treatment_data['user_count'], control_data['user_count']]

# Run the statistical hypothesis test
z_stat, p_value = proportions_ztest(successes, trials)

print("\n--- Python Statistical Test ---")
print(f"P-Value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Statistically significant. The new UI workflow changed conversion rates.")
else:
    print("Result: Not significant. The difference is just statistical noise, do not launch the new feature.")

### 4. Data Visualization
Generating a bar chart to visualize the final conversion metrics for stakeholders.

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 5))

ax = sns.barplot(x='group', y='conversion_rate', data=summary_df, palette=['#ff9999', '#66b3ff'])
ax.set(title='Checkout Conversion Rate: Control vs. Treatment', xlabel='Test Group', ylabel='Conversion Rate (%)')

# Add 15% extra space at the top so the text doesn't hit the ceiling
ax.margins(y=0.15)

# Loop through all containers to ensure BOTH bars get their labels
for container in ax.containers:
    ax.bar_label(container, fmt='%g%%', padding=3, weight='bold')

plt.show()